## <span style="color:#b8860b">Goodreads books data analysis</span>

An analysis of Goodreads book data collected on **Hugging Face Hub**.

### <span style="color:#b8860b">Packages</span>

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Annotated
from urllib.parse import urlsplit

import numpy as np
import polars as pl
from dotenv import load_dotenv
from huggingface_hub import HfApi
from loguru import logger
from zenml import step
from zenml.client import Client
from zenml.integrations.polars.materializers.dataframe_materializer import (
    PolarsMaterializer,
)

load_dotenv(override=True)

### <span style="color:#b8860b">Configuration</span>

In [ ]:
# --- Goodreads Hugging Face repo information ------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")
HF_REPO_ID = "pfaha/goodreads-books"
HF_REPO_TYPE = "dataset"
HF_RAW_DIR = "raw"

# --- Goodreads Hugging Face repo information ------------------------------------------------
MAX_FILES = 30  # We will only read the first 30 parquet files found during matching with 'PARQUET_FILENAME_PATTERN'

# --- Regex to match needed data files in the Hugging Face repo ------------------------------
PARQUET_FILENAME_PATTERN = re.compile(r"books-part(\d+)\.parquet$")

# --- Grouping and ranking keys needed to update books information through the raw data ------
# Those columns were chosen empirically, and maybe there exists some which are better than them to perform the update ...
GROUPING_KEYS = ["title", "language", "format"]
RANKING_KEYS = ["ratings_count", "reviews_count"]

# --- Ip address to start ZenML zerver -------------------------------------------------------
ZENML_IP_ADDRESS = "127.0.0.1"

# Seed for the NumPy pseudo-random number generator to ensure reproducible splits
RANDOM_STATE = 123

# --- Max validation and testing proportions needed (values between 0 and 1) -----------------
MIN_VAL_PROPORTION, MAX_VAL_PROPORTION = 0.1, 0.2
MIN_TEST_PROPORTION, MAX_TEST_PROPORTION = 0.1, 0.2

# --- Main analysis column -------------------------------------------------------------------
TARGET_COLUMN = "average_rating"

### <span style="color:#b8860b">Start the local ZenML server</span>

In [ ]:
def start_zenml_server(ip_address: str = ZENML_IP_ADDRESS) -> subprocess.Popen:
    """Start a loopback-only ZenML server in a separate Windows console."""
    url = urlsplit(os.environ["ZENML_STORE_URL"])

    if (
        url.scheme != "http"
        or url.hostname != ip_address
        or url.username is not None
        or url.password is not None
        or url.path not in ("", "/")
        or url.query
        or url.fragment
    ):
        raise ValueError(f"ZENML_STORE_URL must be http://{ip_address}:<port>")

    port = url.port
    if port is None or not 1 <= port <= 65535:
        raise ValueError("ZENML_STORE_URL must include a valid port")

    executable = Path(sys.executable).with_name("zenml.exe")
    if not executable.is_file():
        raise FileNotFoundError(f"ZenML executable not found: {executable}")

    server_env = {
        key: value
        for key, value in os.environ.items()
        if not key.upper().startswith("ZENML_STORE_") and key.upper() != "ZENML_ACTIVE_PROJECT_ID"
    }

    return subprocess.Popen(
        [
            str(executable),
            "login",
            "--local",
            "--blocking",
            "--ip-address",
            ip_address,
            "--port",
            str(port),
        ],
        env=server_env,
        creationflags=subprocess.CREATE_NEW_CONSOLE,
    )

In [ ]:
server_process = start_zenml_server()

In [ ]:
client = Client()

assert os.environ["ZENML_STORE_URL"] == client.zen_store.config.url, (
    "Mismatch on the ZEN_STORE_URL !"
)

### <span style="color:#b8860b">BookReader</span>

Here is all needed to read already scraped and saved raw data from previous ETL process.

Public entry point: `read_parquet_files()`.

In [ ]:
class BookReader:
    """Read and concatenate book-data Parquet checkpoints from the Hub.

    Matching checkpoints are selected beneath the configured repository
    directory, downloaded to the local Hugging Face cache, and combined
    into a Polars DataFrame.

    Attributes:
        logger (loguru.Logger): Logger bound to the BookReader component.
        api (HfApi): Hugging Face Hub client configured with the supplied
            authentication token.
    """

    def __init__(self, token: str | None = HF_TOKEN) -> None:
        """Initialize the component logger and Hugging Face Hub client.

        Args:
            token (str | None): Authentication token passed to the Hub
                client. Defaults to HF_TOKEN.

        Returns:
            None: Initializes the reader instance.
        """
        self.logger = logger.bind(component="BookReader")
        self.api = HfApi(token=token)

    def read_parquet(
        self,
        num_files: int,
        repo_id: str,
        repo_type: str,
        raw_dir: str,
    ) -> pl.DataFrame:
        """Download and concatenate matching book-data Parquet checkpoints.

        List repository files and retain paths beneath raw_dir that match
        PARQUET_FILENAME_PATTERN. Select files using [:num_files], download
        them through the Hub client, and concatenate their contents using
        Polars' diagonal_relaxed strategy.

        File selection preserves the order returned by list_repo_files;
        checkpoint filenames are not sorted numerically before selection.

        Args:
            num_files (int): Slice limit applied to matching checkpoint
                paths. Positive values select up to that many files; zero
                selects none. Negative values follow Python slicing
                semantics and exclude files from the end of the matching
                list.
            repo_id (str): Hugging Face repository identifier, typically
                in the form "namespace/repository". Defaults to HF_REPO_ID.
            repo_type (str): Repository type passed to the Hub client,
                such as "dataset", "model", or "space".
            raw_dir (str): Repository-relative directory beneath which
                matching checkpoints are selected. Supply it without a
                trailing slash. Matching paths may include subdirectories.

        Returns:
            pl.DataFrame: Combined contents of the selected checkpoints.
                Columns are combined with how="diagonal_relaxed". Returns
                an empty DataFrame with no columns if no files are selected.

        Notes:
            Repository listing, download, Parquet-reading, and
            concatenation errors propagate to the caller. An empty
            DataFrame is returned only when file selection is empty,
            not as a fallback for failed I/O or invalid data.
        """
        self.logger.info(f"Listing files in {repo_id} ({repo_type})...")
        all_files = self.api.list_repo_files(
            repo_id=repo_id,
            repo_type=repo_type,
        )

        parquet_files = [
            f
            for f in all_files
            if f.startswith(f"{raw_dir}/") and PARQUET_FILENAME_PATTERN.search(f)
        ][:num_files]

        if not parquet_files:
            self.logger.warning(
                f"No 'books-part*.parquet' files found under {raw_dir}/ "
                f"in {repo_id} -- returning an empty DataFrame"
            )
            return pl.DataFrame()

        self.logger.info(f"Found {len(parquet_files)} matching part file(s); downloading...")

        local_paths = []
        for f in parquet_files:
            local_path = self.api.hf_hub_download(
                repo_id=repo_id,
                repo_type=repo_type,
                filename=f,
            )
            local_paths.append(local_path)

        df = pl.concat(
            [pl.read_parquet(path) for path in local_paths],
            how="diagonal_relaxed",
        )

        self.logger.info(f"Loaded {df.height} row(s) across {len(local_paths)} file(s)")

        return df


@step(output_materializers=PolarsMaterializer)
def read_books_step(
    num_files: int = MAX_FILES,
    repo_id: str = HF_REPO_ID,
    repo_type: str = HF_REPO_TYPE,
    raw_dir: str = HF_RAW_DIR,
) -> pl.DataFrame:
    """Read book-data checkpoints through a ZenML step.

    Instantiate BookReader with its default authentication configuration
    and delegate checkpoint selection, downloading, and concatenation to
    read_parquet. Declare PolarsMaterializer for the step output.

    Args:
        num_files (int): Slice limit applied to matching checkpoint paths.
            Positive values select up to that many files; zero selects
            none. Negative values exclude files from the end of the
            matching list. Defaults to MAX_FILES.
        repo_id (str): Hugging Face repository identifier to read from.
            Defaults to HF_REPO_ID.
        repo_type (str): Repository type passed to the Hub client.
            Defaults to HF_REPO_TYPE.
        raw_dir (str): Repository-relative directory containing the
            checkpoints, without a trailing slash. Matching paths may
            include subdirectories. Defaults to HF_RAW_DIR.

    Returns:
        pl.DataFrame: Concatenated contents of the selected checkpoints,
            or an empty DataFrame with no columns if no files are
            selected. This is the underlying function's return value.

    Notes:
        File selection preserves repository-listing order rather than
        numerical checkpoint order. Exceptions raised by BookReader
        propagate to the step execution.
    """
    reader = BookReader()

    return reader.read_parquet(
        num_files=num_files,
        repo_id=repo_id,
        repo_type=repo_type,
        raw_dir=raw_dir,
    )

In [ ]:
raw_books_df = read_books_step()

In [ ]:
with pl.Config(
    tbl_cols=-1,  # show all columns, no truncation
    tbl_rows=-1,  # show all rows requested (not needed for head(2), but harmless)
    fmt_str_lengths=1_000,  # show up to 1,000 chars per string cell before truncating
    tbl_width_chars=-1,  # don't wrap/truncate based on terminal width
):
    print(raw_books_df.head(3))

In [ ]:
raw_books_df.columns

Since our webscraping on Goodreads can generate duplicate for a same book since its statistics such as 'ratings_count' and/or 'reviews_count' can increase in time, we need to remove those eventual duplicate by keeping the latest information for each book.

### <span style="color:#b8860b">BookUpdater</span>

We update previoulsy loaded data using this BookUpdater.

Public entry point: `update_df(df)`.

In [ ]:
class BookUpdater:
    """Deduplicate book records using configurable grouping and ranking columns.

    Treat each unique combination of grouping-column values as one logical
    book. Retain the highest-ranked row within each group, carrying its
    remaining column values into the result.

    Attributes:
        logger (loguru.Logger): Logger bound to the BookUpdater component.
    """

    def __init__(self) -> None:
        """Initialize the component logger.

        Args:
            None.

        Returns:
            None: Initializes the updater instance.
        """
        self.logger = logger.bind(component="BookUpdater")

    def update_df(
        self,
        df: pl.DataFrame,
        grouping_keys: list[str],
        ranking_keys: list[str],
    ) -> pl.DataFrame:
        """Retain the highest-ranked record for each logical book.

        Sort rows by ranking_keys in descending priority order, placing
        null ranking values last. Group the sorted rows by grouping_keys
        and retain the first row from each group.

        Ranking is lexicographic: the first ranking column determines
        priority, and subsequent ranking columns break ties in order.
        Values from the retained row are carried into the output rather
        than aggregated across editions.

        Args:
            df (pl.DataFrame): Input book records containing all columns
                named in grouping_keys and ranking_keys.
            grouping_keys (list[str]): Column names whose combined values
                identify a logical book.
            ranking_keys (list[str]): Column names used to rank records
                within each group, in descending order of priority.
                Higher values rank first, and null values rank last.

        Returns:
            pl.DataFrame: Deduplicated records containing one retained
                row per unique combination of grouping-key values.
                Groups follow their first-appearance order in the
                ranking-sorted input. Non-grouping columns contain the
                values from the first row in each group.

        Notes:
            Missing required columns are logged as a warning, but
            processing continues. Sorting and grouping errors propagate
            to the caller; the warning does not provide a fallback.

            No additional tie-breaker is specified when all ranking
            values are equal, so this method does not guarantee which
            tied row is retained.

            The default grouping and ranking lists are passed through
            without being modified.
        """
        missing_cols = [col for col in grouping_keys + ranking_keys if col not in df.columns]

        if missing_cols:
            self.logger.warning(f"Missing expected column(s): {missing_cols}")

        self.logger.info(f"Deduplicating {df.height} row(s) by {grouping_keys}...")

        updated_df = (
            df.sort(
                by=ranking_keys,
                descending=[True] * len(ranking_keys),
                nulls_last=True,
            )
            .group_by(grouping_keys, maintain_order=True)
            .first()
        )

        self.logger.info(
            f"Kept {updated_df.height} row(s) out of {df.height} "
            f"({df.height - updated_df.height} duplicate(s) removed)"
        )

        return updated_df


@step(output_materializers=PolarsMaterializer)
def update_books_step(
    df: pl.DataFrame,
    grouping_keys: list[str] = GROUPING_KEYS,
    ranking_keys: list[str] = RANKING_KEYS,
) -> pl.DataFrame:
    """Deduplicate book records through a ZenML step.

    Instantiate BookUpdater and delegate record ranking and deduplication
    to update_df. Declare PolarsMaterializer for the step output.

    Args:
        df (pl.DataFrame): Input book records containing all columns
            named in grouping_keys and ranking_keys.
        grouping_keys (list[str]): Column names whose combined values
            identify a logical book. Defaults to GROUPING_KEYS.
        ranking_keys (list[str]): Column names used to rank records
            within each group, in descending order of priority.
            Higher values rank first, and null values rank last.
            Defaults to RANKING_KEYS.

    Returns:
        pl.DataFrame: Deduplicated records containing one retained row
            per unique combination of grouping-key values. Non-grouping
            values come from that group's highest-ranked row. This is
            the underlying function's return value.

    Notes:
        Missing required columns produce a warning in BookUpdater but
        do not stop execution immediately. Exceptions raised during
        sorting or grouping propagate to the step execution.

        No additional tie-breaker is specified for records with equal
        values across every ranking column.
    """
    updater = BookUpdater()

    return updater.update_df(
        df=df,
        grouping_keys=grouping_keys,
        ranking_keys=ranking_keys,
    )

In [ ]:
updated_books_df = update_books_step(df=raw_books_df)

### <span style="color:#b8860b">BookSplitter</span>

We use it on the previous updated data.

Public entry point: `split_df(df)`.

In [ ]:
class BookSplitter:
    """Split book records into random training, validation, and test sets.

    Shuffle row indices using RANDOM_STATE, then assign distinct index
    slices to the test, validation, and training sets. All input columns,
    including the target column, are retained in every output.

    Attributes:
        logger (loguru.Logger): Logger bound to the BookSplitter component.
    """

    def __init__(self) -> None:
        """Initialize the component logger.

        Returns:
            None: Initializes the splitter instance.
        """
        self.logger = logger.bind(component="BookSplitter")

    def split_df(
        self,
        df: pl.DataFrame,
        val_proportion: float,
        test_proportion: float,
        target_column: str,
    ) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
        """Split book records into training, validation, and test sets.

        Validate the input and requested proportions, then generate a
        shuffled array of row indices using RANDOM_STATE. Allocate the
        first indices to the test set, the next indices to the validation
        set, and all remaining indices to the training set.

        Validation and test proportions are fractions of the complete
        input dataset, not fractions of a progressively reduced dataset.
        Their row counts are calculated using Python's round function.

        Args:
            df (pl.DataFrame): Non-empty input book records containing
                target_column. The input DataFrame is not modified.
            val_proportion (float): Fraction of all input rows allocated
                to validation, subject to rounding. Must be within the
                inclusive interval from MIN_VAL_PROPORTION to
                MAX_VAL_PROPORTION.
            test_proportion (float): Fraction of all input rows allocated
                to testing, subject to rounding. Must be within the
                inclusive interval from MIN_TEST_PROPORTION to
                MAX_TEST_PROPORTION.
            target_column (str): Target column whose presence is required
                before splitting. The column is retained in each output;
                its values are not used to determine the split.

        Returns:
            tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]: Training,
                validation, and test DataFrames, in that order. Each
                output retains the input schema. Every input row position
                is assigned to exactly one output.

        Raises:
            ValueError: If target_column is missing, the input is empty,
                either proportion is outside its allowed interval, or
                rounding would leave any output split empty.

        Notes:
            The random generator is initialized with RANDOM_STATE on
            every call. Repeated calls with the same input row order,
            parameters, and random-generator behavior produce the same
            row assignments.

            Splitting is row-based, not stratified, grouped, or temporal.
            It does not ensure target-distribution balance or keep
            related editions in the same split.

            The actual validation and test fractions may differ slightly
            from the requested proportions because row counts are rounded.
        """
        if target_column not in df.columns:
            raise ValueError(f"Target column '{target_column}' is missing from the DataFrame.")

        if df.is_empty():
            raise ValueError("Cannot split an empty DataFrame.")

        if not MIN_VAL_PROPORTION <= val_proportion <= MAX_VAL_PROPORTION:
            raise ValueError(
                "`val_proportion` must be in the interval "
                f"[{MIN_VAL_PROPORTION}, {MAX_VAL_PROPORTION}], "
                f"got {val_proportion}."
            )

        if not MIN_TEST_PROPORTION <= test_proportion <= MAX_TEST_PROPORTION:
            raise ValueError(
                "`test_proportion` must be in the interval "
                f"[{MIN_TEST_PROPORTION}, {MAX_TEST_PROPORTION}], "
                f"got {test_proportion}."
            )

        n_rows = df.height
        n_test = round(n_rows * test_proportion)
        n_val = round(n_rows * val_proportion)
        n_train = n_rows - n_test - n_val

        if min(n_train, n_val, n_test) <= 0:
            raise ValueError(
                "The requested proportions must produce non-empty "
                "training, validation, and test sets. "
                f"Calculated sizes: train={n_train}, "
                f"validation={n_val}, test={n_test}."
            )

        rng = np.random.default_rng(RANDOM_STATE)
        shuffled_indices = rng.permutation(n_rows)

        test_indices = shuffled_indices[:n_test]
        val_indices = shuffled_indices[n_test : n_test + n_val]
        train_indices = shuffled_indices[n_test + n_val :]

        train_df = df[train_indices.tolist(), :]
        val_df = df[val_indices.tolist(), :]
        test_df = df[test_indices.tolist(), :]

        self.logger.info(
            "Created random train/val/test split: "
            f"{train_df.height:,} train rows, "
            f"{val_df.height:,} validation rows, and "
            f"{test_df.height:,} test rows."
        )

        return train_df, val_df, test_df


@step(
    output_materializers={
        "train_df": PolarsMaterializer,
        "val_df": PolarsMaterializer,
        "test_df": PolarsMaterializer,
    }
)
def split_books_step(
    df: pl.DataFrame,
    val_proportion: float = MAX_VAL_PROPORTION,
    test_proportion: float = MIN_TEST_PROPORTION,
    target_column: str = TARGET_COLUMN,
) -> tuple[
    Annotated[pl.DataFrame, "train_df"],
    Annotated[pl.DataFrame, "val_df"],
    Annotated[pl.DataFrame, "test_df"],
]:
    """Split book records into three named ZenML output artifacts.

    Delegate input validation, random index shuffling, and row selection
    to BookSplitter.split_df. Return each resulting DataFrame explicitly
    so ZenML recognizes three separate outputs.

    Args:
        df (pl.DataFrame): Non-empty book records containing target_column.
        val_proportion (float): Fraction of all input rows allocated to
            validation, subject to rounding. Must be within the configured
            validation bounds. Defaults to MAX_VAL_PROPORTION.
        test_proportion (float): Fraction of all input rows allocated to
            testing, subject to rounding. Must be within the configured
            test bounds. Defaults to MIN_TEST_PROPORTION.
        target_column (str): Required target column retained in every
            output. Defaults to TARGET_COLUMN.

    Returns:
        tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]: Training,
            validation, and test DataFrames, in that order. ZenML
            registers them as train_df, val_df, and test_df.

    Raises:
        ValueError: If the target column is missing, the input is empty,
            a proportion is outside its configured bounds, or the
            calculated sizes would leave an output split empty.
    """
    splitter = BookSplitter()

    train_df, val_df, test_df = splitter.split_df(
        df=df,
        val_proportion=val_proportion,
        test_proportion=test_proportion,
        target_column=target_column,
    )

    return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = split_books_step(df=updated_books_df)

### <span style="color:#b8860b">An analysis on data from the training set</span>

In [ ]:
with pl.Config(
    tbl_cols=-1,  # show all columns, no truncation
    tbl_rows=-1,  # show all rows requested (not needed for head(2), but harmless)
    fmt_str_lengths=1_000,  # show up to 1,000 chars per string cell before truncating
    tbl_width_chars=-1,  # don't wrap/truncate based on terminal width
):
    print(train_df.head(3))

We will be analyzing all values in each column of the training data.